# Limpeza e Tratamento de Dados - Censo Escolar 2025

Limpeza e tratamento de dados - Censo Escolar 2025


In [1]:
import pandas as pd

## 1. Carregar o CSV como DataFrame

In [2]:
df = pd.read_csv("data/dataset_escolas_censo2025.csv")

## 2. Inspeção inicial

In [3]:
print("=== INFO GERAL ===")
df.info()

print("\n=== PRIMEIRAS LINHAS ===")
print(df.head())

print("\n=== ESTATÍSTICAS DAS COLUNAS NUMÉRICAS ===")
print(df.describe())

=== INFO GERAL ===
<class 'pandas.DataFrame'>
RangeIndex: 214192 entries, 0 to 214191
Data columns (total 77 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   NU_ANO_CENSO                    214192 non-null  int64  
 1   NO_REGIAO                       214191 non-null  str    
 2   NO_UF                           214191 non-null  str    
 3   SG_UF                           214191 non-null  str    
 4   NO_MUNICIPIO                    214191 non-null  str    
 5   CO_MUNICIPIO                    214192 non-null  int64  
 6   NO_ENTIDADE                     214192 non-null  str    
 7   CO_ENTIDADE                     214192 non-null  int64  
 8   TP_DEPENDENCIA                  214192 non-null  int64  
 9   TP_CATEGORIA_ESCOLA_PRIVADA     42454 non-null   float64
 10  TP_LOCALIZACAO                  214192 non-null  int64  
 11  TP_SITUACAO_FUNCIONAMENTO       214192 non-null  int64  
 12  IN_AGUA_

       NU_ANO_CENSO  CO_MUNICIPIO   CO_ENTIDADE  TP_DEPENDENCIA  \
count      214192.0  2.141920e+05  2.141920e+05   214192.000000   
mean         2025.0  3.050211e+06  3.051589e+07        3.078159   
std             0.0  9.517342e+05  9.481242e+06        0.637038   
min          2025.0  1.100015e+06  1.100002e+07        1.000000   
25%          2025.0  2.313757e+06  2.346196e+07        3.000000   
50%          2025.0  3.119856e+06  3.114342e+07        3.000000   
75%          2025.0  3.548500e+06  3.523735e+07        3.000000   
max          2025.0  5.300108e+06  5.308803e+07        4.000000   

       TP_CATEGORIA_ESCOLA_PRIVADA  TP_LOCALIZACAO  TP_SITUACAO_FUNCIONAMENTO  \
count                 42454.000000   214192.000000              214192.000000   
mean                      1.673388        1.326693                   1.175189   
std                       1.198024        0.469005                   0.425033   
min                       1.000000        1.000000                   1.0

## 3. Remover duplicatas (código único de escola)

In [4]:
duplicadas = df.duplicated(subset="CO_ENTIDADE").sum()
print(f"\nLinhas duplicadas: {duplicadas}")
df = df.drop_duplicates(subset="CO_ENTIDADE")


Linhas duplicadas: 0


## 4. Padronizar nomes de colunas

In [5]:
df.columns = df.columns.str.lower()

## 5. Tratar valores ausentes

In [6]:
print("\n=== COLUNAS COM NaN ===")
nan_por_coluna = df.isna().sum()
print(nan_por_coluna[nan_por_coluna > 0])

# 5.1 NaN em região/UF/município: erro pontual de preenchimento -> remove linha
colunas_identificacao = ["no_regiao", "no_uf", "sg_uf", "no_municipio"]
df = df.dropna(subset=colunas_identificacao)

# 5.2 NaN em infraestrutura: ocorre só em escolas paralisadas/extintas
# (situação de funcionamento 2 ou 3) -> preenche com 0
colunas_infraestrutura = [
    "in_agua_potavel", "in_energia_rede_publica", "in_esgoto_rede_publica",
    "in_banheiro", "in_banheiro_pne", "in_biblioteca", "in_biblioteca_sala_leitura",
    "in_cozinha", "in_laboratorio_ciencias", "in_laboratorio_informatica",
    "in_patio_coberto", "in_patio_descoberto", "in_parque_infantil",
    "in_quadra_esportes", "in_refeitorio", "in_acessibilidade_corrimao",
    "in_acessibilidade_rampas", "in_computador", "in_internet",
    "in_internet_alunos", "in_banda_larga", "in_alimentacao",
    "qt_salas_utilizadas_dentro", "qt_salas_utilizadas_fora", "qt_salas_utilizadas",
    "qt_salas_utiliza_climatizadas", "qt_salas_utilizadas_acessiveis", "qt_salas_leitura",
    "qt_equip_dvd", "qt_equip_som", "qt_equip_tv", "qt_equip_lousa_digital",
    "qt_equip_multimidia", "qt_desktop_aluno", "qt_comp_portatil_aluno", "qt_tablet_aluno",
    "qt_prof_administrativos", "qt_prof_servicos_gerais", "qt_prof_bibliotecario",
    "qt_prof_saude", "qt_prof_coordenador", "qt_prof_psicologo", "qt_prof_alimentacao",
    "qt_prof_pedagogia", "qt_prof_secretario", "qt_prof_seguranca", "qt_prof_monitores",
    "qt_prof_gestao", "qt_prof_assist_social",
]
df[colunas_infraestrutura] = df[colunas_infraestrutura].fillna(0)

# 5.3 NaN em tp_categoria_escola_privada: só existe para escola privada -> -1 (não aplicável)
df["tp_categoria_escola_privada"] = df["tp_categoria_escola_privada"].fillna(-1)

print(f"\nTotal de NaN restantes: {df.isna().sum().sum()}")


=== COLUNAS COM NaN ===
no_regiao                              1
no_uf                                  1
sg_uf                                  1
no_municipio                           1
tp_categoria_escola_privada       171738
in_agua_potavel                    33652
in_energia_rede_publica            33652
in_esgoto_rede_publica             33652
in_banheiro                        33652
in_banheiro_pne                    33652
in_biblioteca                      33652
in_biblioteca_sala_leitura         33652
in_cozinha                         33652
in_laboratorio_ciencias            33652
in_laboratorio_informatica         33652
in_patio_coberto                   33652
in_patio_descoberto                33652
in_parque_infantil                 33652
in_quadra_esportes                 33652
in_refeitorio                      33652
in_acessibilidade_corrimao         33652
in_acessibilidade_rampas           33652
qt_salas_utilizadas_dentro         33652
qt_salas_utilizadas_fora        


Total de NaN restantes: 0


## 6. Corrigir tipos (float -> int, já que são contagens/categorias)

In [7]:
colunas_para_int = colunas_infraestrutura + ["tp_categoria_escola_privada"]
df[colunas_para_int] = df[colunas_para_int].astype(int)

## 7. Remover coluna redundante (tem_matricula duplica qt_mat_bas)

In [8]:
df = df.drop(columns=["tem_matricula"])

## 8. Exportar dataset limpo

In [9]:
print("\n=== INFO FINAL ===")
df.info()

df.to_csv("data/dataset_escolas_censo2025_limpo.csv", index=False)
print("\nArquivo salvo em data/dataset_escolas_censo2025_limpo.csv")


=== INFO FINAL ===
<class 'pandas.DataFrame'>
Index: 214191 entries, 0 to 214191
Data columns (total 76 columns):
 #   Column                          Non-Null Count   Dtype
---  ------                          --------------   -----
 0   nu_ano_censo                    214191 non-null  int64
 1   no_regiao                       214191 non-null  str  
 2   no_uf                           214191 non-null  str  
 3   sg_uf                           214191 non-null  str  
 4   no_municipio                    214191 non-null  str  
 5   co_municipio                    214191 non-null  int64
 6   no_entidade                     214191 non-null  str  
 7   co_entidade                     214191 non-null  int64
 8   tp_dependencia                  214191 non-null  int64
 9   tp_categoria_escola_privada     214191 non-null  int64
 10  tp_localizacao                  214191 non-null  int64
 11  tp_situacao_funcionamento       214191 non-null  int64
 12  in_agua_potavel                 214191 n


Arquivo salvo em data/dataset_escolas_censo2025_limpo.csv
